In [1]:
import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

In [2]:
from getpass import getpass

password = getpass("Enter PostgreSQL password: ")

url = URL.create(
    "postgresql+psycopg2",
    username="postgres",
    password=password,
    host="localhost",
    port=5432,
    database="malaysian_fnb_bi",
)

engine = create_engine(url)

In [3]:
with engine.connect() as conn:
    result = conn.execute(text("SELECT current_database()"))

    print(result.scalar())

malaysian_fnb_bi


In [4]:
sales = pd.read_csv("../data/processed/sales_clean.csv")
products = pd.read_csv("../data/processed/products_clean.csv")
outlets = pd.read_csv("../data/processed/outlets_clean.csv")
customers = pd.read_csv("../data/processed/customers_clean.csv")
payments = pd.read_csv("../data/processed/payments_clean.csv")

print("Sales:", sales.shape)
print("Products:", products.shape)
print("Outlets:", outlets.shape)
print("Customers:", customers.shape)
print("Payments:", payments.shape)

Sales: (118744, 14)
Products: (30, 5)
Outlets: (15, 4)
Customers: (8000, 2)
Payments: (118744, 3)


In [5]:
print(sales.columns.tolist())

['transaction_id', 'transaction_datetime', 'outlet_id', 'product_id', 'customer_id', 'quantity', 'unit_price', 'unit_cost', 'discount_rate', 'gross_sales', 'discount_amount', 'net_sales', 'product_cost', 'gross_profit']


In [6]:
print("Duplicate sales IDs:", sales["transaction_id"].duplicated().sum())
print("Duplicate product IDs:", products["product_id"].duplicated().sum())
print("Duplicate payment IDs:", payments["transaction_id"].duplicated().sum())

Duplicate sales IDs: 0
Duplicate product IDs: 0
Duplicate payment IDs: 0


In [7]:
sales["quantity"] = pd.to_numeric(sales["quantity"], errors="raise")

assert sales["quantity"].notna().all()
assert (sales["quantity"] > 0).all()
assert (sales["quantity"] % 1 == 0).all()

sales["quantity"] = sales["quantity"].astype(int)

print(sales["quantity"].dtype)

int64


In [8]:
sales["transaction_datetime"] = pd.to_datetime(
    sales["transaction_datetime"], errors="raise"
)

print(sales["transaction_datetime"].dtype)

datetime64[us]


In [9]:
print(
    "Invalid customers:", (~sales["customer_id"].isin(customers["customer_id"])).sum()
)
print("Invalid products:", (~sales["product_id"].isin(products["product_id"])).sum())
print("Invalid outlets:", (~sales["outlet_id"].isin(outlets["outlet_id"])).sum())
print(
    "Invalid payment references:",
    (~payments["transaction_id"].isin(sales["transaction_id"])).sum(),
)

Invalid customers: 0
Invalid products: 0
Invalid outlets: 0
Invalid payment references: 0


In [10]:
for name, df in {
    "sales": sales,
    "products": products,
    "outlets": outlets,
    "customers": customers,
    "payments": payments,
}.items():
    print(name, "missing cells:", df.isna().sum().sum())

sales missing cells: 0
products missing cells: 5
outlets missing cells: 0
customers missing cells: 65
payments missing cells: 60


In [11]:
tables = ["dim_products", "dim_outlets", "dim_customers", "fact_sales", "fact_payments"]

for table in tables:
    query = text(f"SELECT COUNT(*) FROM {table}")

    with engine.connect() as conn:
        count = conn.execute(query).scalar()

    print(table, count)

dim_products 30
dim_outlets 15
dim_customers 8000
fact_sales 118744
fact_payments 118744


In [12]:
with engine.begin() as conn:
    conn.execute(text("""
        TRUNCATE TABLE
            fact_payments,
            fact_sales,
            dim_products,
            dim_outlets,
            dim_customers
        RESTART IDENTITY;
    """))

print("Project tables cleared successfully.")

Project tables cleared successfully.


In [13]:
products = products[
    ["product_id", "product_name", "category", "unit_price", "unit_cost"]
]

outlets = outlets[["outlet_id", "outlet_name", "state", "city"]]

customers = customers[["customer_id", "customer_segment"]]

In [14]:
sales = sales[
    [
        "transaction_id",
        "transaction_datetime",
        "outlet_id",
        "customer_id",
        "product_id",
        "quantity",
        "unit_price",
        "unit_cost",
        "discount_rate",
        "gross_sales",
        "discount_amount",
        "net_sales",
        "product_cost",
        "gross_profit",
    ]
]

payments = payments[["transaction_id", "payment_method", "payment_amount"]]

In [ ]:
with engine.begin() as conn:

    products.to_sql("dim_products", conn, if_exists="append", index=False)

    outlets.to_sql("dim_outlets", conn, if_exists="append", index=False)

    customers.to_sql("dim_customers", conn, if_exists="append", index=False)

    sales.to_sql("fact_sales", conn, if_exists="append", index=False, chunksize=1000)

    payments.to_sql(
        "fact_payments", conn, if_exists="append", index=False, chunksize=1000
    )

All five tables loaded successfully!


In [16]:
expected = {
    "dim_products": len(products),
    "dim_outlets": len(outlets),
    "dim_customers": len(customers),
    "fact_sales": len(sales),
    "fact_payments": len(payments),
}

with engine.connect() as conn:
    for table, expected_count in expected.items():
        actual = conn.execute(text(f"SELECT COUNT(*) FROM {table}")).scalar()

        status = "PASS" if actual == expected_count else "FAIL"

        print(table, actual, status)

        assert actual == expected_count

dim_products 30 PASS
dim_outlets 15 PASS
dim_customers 8000 PASS
fact_sales 118744 PASS
fact_payments 118744 PASS


In [17]:
query = "SELECT * FROM fact_sales LIMIT 5"

df = pd.read_sql(query, engine)

df

,transaction_id,transaction_datetime,outlet_id,customer_id,product_id,quantity,unit_price,unit_cost,discount_rate,gross_sales,discount_amount,net_sales,product_cost,gross_profit
0,T000001,2024-02-08 11:27:00,O002,C07750,P029,1,12.9,5.40,0.00,12.9,0.00,12.90,5.40,7.50
1,T000002,2024-11-25 21:30:00,O005,C03915,P030,1,10.9,4.30,0.15,10.9,1.64,9.26,4.30,4.96
2,T000003,2024-08-28 14:30:00,O002,C02014,P024,2,7.9,3.10,0.00,15.8,0.00,15.80,6.20,9.60
3,T000004,2024-02-25 16:19:00,O004,C01999,P004,3,11.9,4.70,0.10,35.7,3.57,32.13,14.10,18.03
4,T000005,2024-09-03 21:00:00,O012,C00090,P014,1,3.9,0.85,0.00,3.9,0.00,3.90,0.85,3.05


In [18]:
query = """
SELECT
    ROUND(SUM(net_sales), 2) AS total_revenue
FROM fact_sales
"""

revenue = pd.read_sql(query, engine)

revenue

,total_revenue
0,1705288.9


In [19]:
round(sales["net_sales"].sum(), 2)

np.float64(1705288.9)

In [20]:
query = """
SELECT
    o.outlet_name,
    ROUND(SUM(s.net_sales), 2) AS revenue
FROM fact_sales s
JOIN dim_outlets o
    ON s.outlet_id = o.outlet_id
GROUP BY o.outlet_name
ORDER BY revenue DESC
"""

outlet_revenue = pd.read_sql(query, engine)

outlet_revenue.head(10)

,outlet_name,revenue
0,Rasa Malaysia - Puchong,114993.02
1,Rasa Malaysia - Petaling Jaya,114904.45
2,Rasa Malaysia - Subang Jaya,114828.29
3,Rasa Malaysia - George Town,114780.65
4,Rasa Malaysia - Bukit Bintang,114498.73
5,Rasa Malaysia - Butterworth,114450.80
6,Rasa Malaysia - Klang,114415.98
7,Rasa Malaysia - Shah Alam,113901.35
8,Rasa Malaysia - Bangsar,113659.59
9,Rasa Malaysia - Bayan Lepas,113378.54


In [21]:
python_revenue = round(sales["net_sales"].sum(), 2)
sql_revenue = round(float(revenue.loc[0, "total_revenue"]), 2)

print("Python revenue:", python_revenue)
print("SQL revenue:", sql_revenue)
print("Difference:", round(python_revenue - sql_revenue, 2))

Python revenue: 1705288.9
SQL revenue: 1705288.9
Difference: 0.0
